# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)

In [3]:
mol_instruction_dataset

DatasetDict({
    description_guided_molecule_design: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 298319
    })
    forward_reaction_prediction: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 125384
    })
    molecular_description_generation: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 298319
    })
    property_prediction: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 362100
    })
    reagent_prediction: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 125384
    })
    retrosynthesis: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 129684
    })
})

In [4]:
forward_data = mol_instruction_dataset['forward_reaction_prediction']

In [10]:
def get_molinst_data_list(
        data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(data)))
    for i in iter_bar:
        data_instance = data[i]
        selfies = data_instance['input']
        if task == 'reagent_prediction':
            selfies, additional_selfies = selfies.split('>>')
            smiles = sf.decoder(selfies)
            mol = Chem.MolFromSmiles(smiles)
            additional_smiles = sf.decoder(additional_selfies)
            additional_mol = Chem.MolFromSmiles(additional_smiles)
            mol = [mol, additional_mol]
        else:
            smiles = sf.decoder(selfies)
            mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [6]:
list_train_data, list_test_data = get_molinst_data_list(
    data=forward_data,
    instruction_templates=instructions_smol.forward_reaction_prediction,
    task="forward_reaction_prediction",
)

  0%|          | 0/125384 [00:00<?, ?it/s][13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
  0%|          | 97/125384 [00:00<02:09, 968.32it/s][13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighbors
[13:26:56] WARNING: not removing hydrogen atom without neighb

124384 1000


In [7]:
data_dict = {
    "train": list_train_data,
    "test": list_test_data
}


for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_forward_reaction_prediction_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 59091.35 examples/s]


In [11]:
retro_data = mol_instruction_dataset['retrosynthesis']
list_retro_train_data, list_retro_test_data = get_molinst_data_list(
    data=retro_data,
    instruction_templates=instructions_smol.retrosynthesis,
    task="retrosynthesis",
)

100%|██████████| 1000/1000 [00:01<00:00, 884.28it/s]

128684 1000


In [12]:
reagent_data = mol_instruction_dataset['reagent_prediction']
list_reagent_train_data, list_reagent_test_data = get_molinst_data_list(
    data=reagent_data,
    instruction_templates=instructions_smol.reagent_prediction,
    task="reagent_prediction",
)

  0%|          | 0/125384 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:02<00:00, 407.49it/s]

124384 1000


In [9]:
mol_instruction_dataset['reagent_prediction'][0]

{'instruction': 'Based on the given chemical reaction, can you propose some likely reagents that might have been utilized?',
 'input': '[O][=C][C][=C][C][Branch1][=Branch1][N+1][=Branch1][C][=O][O-1][=C][Branch1][C][F][C][=C][Ring1][#Branch2][F]>>[O][=N+1][Branch1][C][O-1][C][=C][C][Branch1][Ring1][C][O][=C][Branch1][C][F][C][=C][Ring1][=Branch2][F]',
 'output': '[C][C][C][O][C][Ring1][Branch1].[Cl].[BH4-1].[Na+1]',
 'metadata': "{'task': 'reagent prediction', 'split': 'train'}"}

In [13]:
list_rxn_train_data = list_train_data + list_retro_train_data + list_reagent_train_data
list_rxn_test_data = list_test_data + list_retro_test_data + list_reagent_test_data

In [21]:
list_reagent_train_data[0]

{'task': 'reagent_prediction',
 'x': array([[7, 0, 1, 5, 0, 0, 1, 0, 0],
        [5, 0, 3, 5, 1, 0, 1, 0, 0],
        [5, 0, 3, 5, 0, 0, 1, 1, 1],
        [5, 0, 3, 5, 1, 0, 1, 1, 1],
        [5, 0, 3, 5, 0, 0, 1, 1, 1],
        [6, 0, 3, 6, 0, 0, 1, 0, 0],
        [7, 0, 1, 5, 0, 0, 1, 0, 0],
        [7, 0, 1, 4, 0, 0, 1, 0, 0],
        [5, 0, 3, 5, 0, 0, 1, 1, 1],
        [8, 0, 1, 5, 0, 0, 2, 0, 0],
        [5, 0, 3, 5, 1, 0, 1, 1, 1],
        [5, 0, 3, 5, 0, 0, 1, 1, 1],
        [8, 0, 1, 5, 0, 0, 2, 0, 0]]),
 'edge_index': array([[ 0,  1,  1,  2,  2,  3,  3,  4,  4,  5,  5,  6,  5,  7,  4,  8,
          8,  9,  8, 10, 10, 11, 11, 12, 11,  2],
        [ 1,  0,  2,  1,  3,  2,  4,  3,  5,  4,  6,  5,  7,  5,  8,  4,
          9,  8, 10,  8, 11, 10, 12, 11,  2, 11]]),
 'edge_attr': array([[1, 0, 1],
        [1, 0, 1],
        [0, 0, 1],
        [0, 0, 1],
        [3, 0, 1],
        [3, 0, 1],
        [3, 0, 1],
        [3, 0, 1],
        [0, 0, 1],
        [0, 0, 1],
        [1, 0, 1

In [19]:
len(list_rxn_train_data), len(list_rxn_test_data)

(377452, 3000)

In [20]:
rxn_dict = {
    "task": "rxn",
    "train": list_rxn_train_data,
    "test": list_rxn_test_data,
    "validation": list_rxn_test_data
}

for split in ["train", "test", "validation"]:
    list_data = rxn_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_rxn_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 82265.45 examples/s]
